# Milestone 1 arXiv Crawler trên Google Colab — v3.18

Cào toàn bộ dải ID (`id_ranges.csv`) trên **1 máy duy nhất** (tuân thủ nghiêm ngặt Quyết định D8 & rubric đồ án).
Lưu `S2_API_KEY` trong mục **Secrets** của Colab (khóa `S2_API_KEY`, không bao giờ hardcode vào notebook).

---

### Kiến trúc 2 giai đoạn & Đột phá kỹ thuật — v3.18
Nhằm tối ưu đồng thời tốc độ tải `4–6 MiB/s`, tránh bóp băng thông khi đụng chuỗi bài nặng, và RAM `O(1)` trên Colab CPU:

1. **Giai đoạn 1 (Tải & Trích xuất In-Flight TeX Source arXiv)**: Dùng `--skip-s2 --http2 --in-flight-extract` dồn 100% tài nguyên cho `export.arxiv.org` với nhịp **Start-to-Start chuẩn `3.05s + jitter 0..0.10s`** (`min_idle_gap = 0.25s`).
   - **Đột phá 1 (HTTP/2 + Zero-Body Early-Reject)**: Kết nối đơn `HTTP/2` (`httpx`), cắt tức thì PDF/ZIP ngay từ `Content-Type` header trước khi đọc byte body và hủy luồng bằng `RST_STREAM` mà không làm đứt TCP+TLS Keep-Alive (log `httpx`/`httpcore` giữ gọn ở mức `WARNING`).
   - **Đột phá 2 (CDN-Aware Telemetry + Preflight POP Scorer + Self-Healing Recovery)**: Đọc `X-Cache`/`X-Served-By` để tách riêng băng thông **Fastly Edge HIT** vs **Origin Cache MISS**, chấm điểm VM Colab (`Grade A/B/C`) trước phiên 5.000 ID, kích Defensive Soft-Pacing nếu `HIT < 3.0 MiB/s` và tự ngủ hồi phục 300s sau 2 cửa sổ liên tiếp bị bóp.
   - **Đột phá 3 (Cut-Through In-Flight Extraction `O(1)` RAM)**: Giải nén 1-pass trực tiếp trên luồng mạng (đệm 128 KiB), lọc bỏ ~95% byte ảnh ngay trên đường bay (0 byte ảnh tạm xuống đĩa; nếu đứt luồng giữa chừng tự động fallback sang `HTTP 206 Range` ở lượt vét).
   - **Đột phá 4 (Traffic-Shaping Interleaver + Sliding Byte-Budget Governor — bật mặc định `v3.17`)**: Giữ nguyên dây tải `4–6 MiB/s` nhưng bẻ gãy các chuỗi bài nặng liên tiếp (tách `v2..vN` của bài `>= 25 MiB` xen giữa ~15 version nhẹ, đảo trải cụm bài nặng cùng ngày theo bước nhảy tỷ lệ vàng trên từng khối `>= 16` bài, và giới hạn ngân sách byte trượt `360 MiB / 180s` để không làm cạn Token Bucket).
   - *Kaggle Snapshot Offline*: Thêm cờ `--snapshot` để nạp sẵn metadata 25.000 bài trong `< 60s`, triệt tiêu 100% Atom API qua mạng.
2. **Giai đoạn 2 (Bổ sung trích dẫn Semantic Scholar — v3.18 High-Density & Save-First)**: Dùng cờ `--s2-only --enable-bib-fallback --s2-batch-size 150` sau khi Giai đoạn 1 hoàn tất (hoặc thêm `--limit 15000` nếu mới cào xong 15.000 ID đầu). Hệ thống gọi S2 Graph API theo lô 150 bài (~3,9 MB, giảm 6× số request), tự chia đôi khi chạm ngưỡng 400/413/5xx/10.000 refs, lưu từng bài ngay khi xong (Save-First Isolation), backoff nhanh `2→4→8→15s` + AIMD hồi phục sau 5 lần `200`, tự vét lại tối đa 2 lượt cuối phiên, và tự fallback sang `.bib/.bbl` nếu S2 404.

---

### Vòng lặp phiên chạy an toàn: Chạy khi rảnh, dừng khi bận — không mất dữ liệu!
Dữ liệu được lưu trữ an toàn trên đĩa cục bộ Colab (`/content/data`) và tự động đồng bộ lên Google Drive (`arxiv-data.tar.gz`):

1. **Setup**: Clone / pull mã nguồn, mount Google Drive, cấu hình tham số.
2. **Restore**: Bung backup `arxiv-data.tar.gz` từ Google Drive vào `/content/data` (chỉ mất vài giây).
3. **Crawl**: Chạy 1 trong 4 cell cào tương ứng bên dưới. Hệ thống tự động đọc `manifest.sqlite`, **bỏ qua 100% các bài/version đã hoàn tất**, chỉ cào phần còn thiếu. Tự động lưu backup nền mỗi 1000 ID và khi dừng.
4. **Dừng an toàn**: Bấm **Runtime → Interrupt execution**, sau đó chạy cell **Backup** (`backup_now(unmount=True)`) trước khi đóng tab.
5. **Phiên tiếp theo**: Lặp lại từ bước 1.


In [ ]:
import os

# Nhánh mặc định trên GitHub (đổi nếu chạy thử trên nhánh phụ)
BRANCH = "main"

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone -b {BRANCH} https://github.com/GiaaPhuc/arxiv-pipeline.git /content/arxiv-pipeline
else:
    !git -C /content/arxiv-pipeline fetch origin
    !git -C /content/arxiv-pipeline checkout {BRANCH}
    !git -C /content/arxiv-pipeline pull --ff-only origin {BRANCH}
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import logging
import os
import sys
import time
from pathlib import Path

from google.colab import drive, userdata

if "/content/arxiv-pipeline/src" not in sys.path:
    sys.path.insert(0, "/content/arxiv-pipeline/src")

from crawler.checkpoint import (
    backup_tar_archive,
    is_workspace_complete,
    manifest_pair_counts,
    read_archive_meta,
    restore_data_dir,
)

_DRIVE_MOUNT = "/content/drive"
_DATA_DIR = Path("/content/data")
_BACKUP_TAR_GZ = Path("/content/drive/MyDrive/arxiv-data.tar.gz")
_BACKUP_TAR = Path("/content/drive/MyDrive/arxiv-data.tar")
_BACKUP_LEGACY_DIR = Path("/content/drive/MyDrive/arxiv-data")
_SNAPSHOT_PATH = Path("/content/drive/MyDrive/arxiv-metadata-oai-snapshot.json")


def _drive_usable() -> bool:
    my_drive = os.path.join(_DRIVE_MOUNT, "MyDrive")
    if not os.path.isdir(my_drive):
        return False
    try:
        os.listdir(my_drive)
        return True
    except OSError:
        return False


if _drive_usable():
    print("Google Drive already mounted.")
else:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(_DRIVE_MOUNT) and os.listdir(_DRIVE_MOUNT):
        raise RuntimeError(
            "Cannot mount Drive: /content/drive still has files "
            "(common after disconnect). Runtime → Restart session, "
            "then run this notebook from the top."
        )
    drive.mount(_DRIVE_MOUNT)

# Configuration & API Secrets
os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # ĐỔI THÀNH EMAIL THẬT (User-Agent arXiv)
os.environ["CRAWLER_DATA_DIR"] = str(_DATA_DIR)
os.environ["CRAWLER_BACKUP_DIR"] = str(_BACKUP_TAR_GZ)
os.environ["CRAWLER_BACKUP_EVERY_IDS"] = "2500"  # Backup mỗi 2500 ID (tránh nghẽn CPU)
os.environ["CRAWLER_ARXIV_MIN_INTERVAL"] = "3.05"  # v3.15: Start-to-Start >= 3.0s
os.environ["CRAWLER_ARXIV_MIN_IDLE_GAP"] = "0.25"  # v3.15: đệm hậu tải 0.25s
os.environ["CRAWLER_ARXIV_JITTER_MAX"] = "0.10"  # v3.15: jitter 0..0.10s (TB ~3.10s)
os.environ["CRAWLER_ARXIV_SOCKET_RECYCLE_EVERY"] = "500"  # v3.15: giữ ấm Keep-Alive
os.environ["CRAWLER_ARXIV_PROACTIVE_COOLDOWN_EVERY_IDS"] = "0"  # 0=disabled
os.environ["CRAWLER_ARXIV_PROACTIVE_COOLDOWN_SECONDS"] = "30"
os.environ["CRAWLER_AUTO_RESUME_RATE_LIMIT"] = "1"  # Auto-defer & cooldown on 429
os.environ["CRAWLER_ENABLE_BIB_FALLBACK"] = "1"  # Dual-engine local .bib fallback
os.environ["CRAWLER_ENABLE_OAI_DATES"] = "1"  # OAI-PMH arXivRaw date recovery
os.environ["CRAWLER_S2_BATCH_SIZE"] = (
    "150"  # v3.18: High-Density batch 150 (~3.9 MB, tự bisection khi >10 MB/10k refs)
)
os.environ["CRAWLER_MAX_STREAM_MBPS"] = "6.0"  # v3.15: mượt 6.0 MiB/s (~7.7h/5000 ID)
os.environ["CRAWLER_USE_HTTP2"] = "1"  # v3.16: HTTP/2 kết nối đơn + RST_STREAM early abort
os.environ["CRAWLER_IN_FLIGHT_EXTRACT"] = "1"  # v3.16: giải nén 1-pass trên luồng (RAM O(1))
os.environ["CRAWLER_TRAFFIC_SHAPING"] = (
    "1"  # v3.17: interleave bài nặng + byte-budget 360 MiB/180s (0 để tắt)
)

if _SNAPSHOT_PATH.is_file():
    print(f"Kaggle Snapshot found on Drive: {_SNAPSHOT_PATH.name} (ready for offline discovery)")
else:
    print("Kaggle Snapshot not found on Drive (will use online arXiv Atom discovery by default).")


def _archive_rank(path: Path) -> tuple[int, float]:
    meta = read_archive_meta(path)
    raw_pairs = meta.get("final_pairs") if meta else None
    pairs = int(raw_pairs) if isinstance(raw_pairs, (int, float)) else -1
    return (pairs, path.stat().st_mtime)


def backup_now(*, allow_shrink: bool = False, unmount: bool = False) -> None:
    """Pack /content/data into one atomic archive on Drive (with shrink guard)."""
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not (_DATA_DIR / "manifest.sqlite").is_file():
        print("Nothing to back up yet (/content/data/manifest.sqlite not found).")
        return
    info = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ, allow_shrink=allow_shrink)
    mib = info["archive_bytes"] / (1024 * 1024)
    print(
        f"backup done {time.strftime('%H:%M:%S')}: {_BACKUP_TAR_GZ.name} "
        f"({mib:.1f} MiB, {info['final_pairs']} final pairs)"
    )
    if unmount:
        drive.flush_and_unmount()
        print("Drive flushed to cloud and unmounted — safe to close Colab.")


def restore_now(*, force: bool = False) -> None:
    """Restore /content/data from Drive archive (or migrate legacy folder once)."""
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not force and is_workspace_complete(_DATA_DIR):
        local_final, _ = manifest_pair_counts(_DATA_DIR / "manifest.sqlite")
        print(
            f"Skip restore: {_DATA_DIR} already has complete local data "
            f"({local_final} final pairs). Pass restore_now(force=True) to overwrite."
        )
        return

    archives = [p for p in (_BACKUP_TAR_GZ, _BACKUP_TAR) if p.is_file()]
    if archives:
        chosen = max(archives, key=_archive_rank)
        info = restore_data_dir(chosen, _DATA_DIR, force=force)
        print(f"Restored {chosen.name} -> {_DATA_DIR} ({info['final_pairs']} final pairs).")
        if chosen != _BACKUP_TAR_GZ:
            backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            print(f"Migrated {chosen.name} -> {_BACKUP_TAR_GZ.name}.")
        return

    has_legacy = _BACKUP_LEGACY_DIR.is_dir() and (
        (_BACKUP_LEGACY_DIR / "manifest.sqlite").is_file()
        or (_BACKUP_LEGACY_DIR / "papers").is_dir()
    )
    if has_legacy:
        print(
            "Legacy folder backup found on Drive: running 16-thread parallel restore "
            "and migrating to arxiv-data.tar.gz..."
        )
        info = restore_data_dir(
            _BACKUP_LEGACY_DIR,
            _DATA_DIR,
            force=force,
            legacy_workers=16,
            progress_every=250,
        )
        if (_DATA_DIR / "manifest.sqlite").is_file():
            bk = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            mib = bk["archive_bytes"] / (1024 * 1024)
            print(
                f"Migrated legacy folder -> {_BACKUP_TAR_GZ.name} "
                f"({mib:.1f} MiB, {info['final_pairs']} final pairs). "
                "Next restore will take seconds!"
            )
        return

    _DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("No backup on Drive (first run).")

In [ ]:
# [TÙY CHỌN NÂNG CAO] Tải Kaggle Snapshot về Google Drive (chỉ cần chạy 1 lần duy nhất)
# Yêu cầu: Đã upload tệp kaggle.json vào MyDrive hoặc thư mục gốc Drive
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d Cornell-University/arxiv -p /content/drive/MyDrive/ --unzip

In [ ]:
# Plan only: kiểm tra danh sách ID và cấu hình (không gọi mạng, không ghi file)
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore from Drive — Chạy MỖI KHI bắt đầu một session Colab mới.
# - Tự động giải nén arxiv-data.tar.gz từ Drive vào /content/data trong vài giây.
# - Dữ liệu cũ đã crawl sẽ được phục hồi nguyên vẹn và KHÔNG BAO GIỜ bị cào lại!
restore_now()

In [ ]:
# Preflight POP & VM Scorer (~15s, 4 bài lạnh): kiểm tra vùng Fastly POP & ước tính giờ/5000 ID
# Nếu ra Grade C trên Colab, hãy Runtime -> Disconnect and delete runtime để đổi sang VM gần hơn!
!python crawler/preflight.py --ranges id_ranges.csv --offset 12000 --count 4 --http2

# [TÙY CHỌN BÁO CÁO] Benchmark A/B HTTP/1.1 vs HTTP/2 trên ID tách rời (xuất JSON cho Jupyter Book):
# !python crawler/preflight.py --mode ab-benchmark --ranges id_ranges.csv \
#   --offset 12500 --count 10 --output /content/data/logs/ab_transport.json

### 4 Chế độ Crawl (Chọn chạy 1 cell tương ứng bên dưới tùy theo nhu cầu)

1. **LOẠI 1: CRAWL TOÀN BỘ + KAGGLE SNAPSHOT + SKIP S2 (LẦN CHẠY ĐẦU TIÊN)**:
   - Dành cho lần chạy đầu tiên khi có tệp `arxiv-metadata-oai-snapshot.json` trên Google Drive.
   - Đọc snapshot offline trong < 60s, nạp sẵn `paper.json` cho 25.000 bài, bỏ qua 100% gọi Atom API qua mạng.
   - Cào gói TeX mượt ở tốc độ 6.0 MiB/s (~7,7h/5.000 ID khi mạng ổn định), bỏ qua S2 để tối đa hóa tốc độ và hạn chế 429.

2. **LOẠI 2: CRAWL TOÀN BỘ KHI ĐÃ CÓ KAGGLE SNAPSHOT LẦN TRƯỚC + SKIP S2 (RESUME CÁC PHIÊN SAU)**:
   - Dành cho các phiên chạy tiếp theo (khi session trước bị ngắt kết nối hoặc tạm dừng).
   - Vì lần trước đã nạp snapshot nên `paper.json` của 25.000 bài đã có sẵn trong `/content/data/papers/`.
   - Không cần nạp lại snapshot; hệ thống tự động cào tiếp các version TeX còn thiếu với `--skip-s2`.

3. **LOẠI 3: ONLY S2 — LÀM GIÀU TRÍCH DẪN SEMANTIC SCHOLAR (v3.18 — BATCH 150)**:
   - Khuyên chạy sau khi Giai đoạn 1 đã hoàn tất TeX (hoặc thêm `--limit 15000` nếu mới cào xong 15.000 ID đầu).
   - Không gọi máy chủ arXiv; gọi S2 Graph API theo lô 150 bài (giảm 6× số request), Save-First lưu ngay từng bài hoàn tất và tự fallback sang `.bib/.bbl` có sẵn.

4. **LOẠI 4: LIMIT 30 — CHẠY THỬ NGHIỆM PILOT (SMOKE TEST)**:
   - Chạy thử 30 bài đầu dải để kiểm tra toàn bộ pipeline từ đầu đến cuối trước khi cào lớn.


In [ ]:
# ==============================================================================
# LOẠI 1: CRAWL TOÀN BỘ + KAGGLE SNAPSHOT + SKIP S2 (LẦN CHẠY ĐẦU TIÊN)
# ==============================================================================
# - Đọc snapshot từ Drive: nạp trước 25.000 metadata offline trong < 60s.
# - Cào gói nguồn TeX mượt (--max-stream-mbps 6.0), mục tiêu ~8h/5000 ID, hạn chế burst CDN.
!python run_crawl.py --ranges id_ranges.csv \
    --snapshot /content/drive/MyDrive/arxiv-metadata-oai-snapshot.json \
    --skip-s2 \
    --auto-resume-rate-limit \
    --max-stream-mbps 6.0 \
    --http2 \
    --in-flight-extract

In [ ]:
# ==============================================================================
# LOẠI 2: CRAWL TOÀN BỘ TIẾP TỤC (ĐÃ NẠP SNAPSHOT TỪ LẦN TRƯỚC) + SKIP S2
# ==============================================================================
# - Dành cho các phiên chạy tiếp theo (resume sau khi tạm dừng hoặc mất kết nối).
# - paper.json đã có sẵn, manifest.sqlite tự động bỏ qua các bài/version đã xong.
!python run_crawl.py --ranges id_ranges.csv \
    --skip-s2 \
    --auto-resume-rate-limit \
    --max-stream-mbps 6.0 \
    --http2 \
    --in-flight-extract

In [ ]:
# ==============================================================================
# LOẠI 3: ONLY S2 — LÀM GIÀU TRÍCH DẪN SEMANTIC SCHOLAR (v3.18 BATCH 150)
# ==============================================================================
# - Không chạm vào arXiv; cào bổ sung S2 theo lô 150 bài + Save-First + fallback .bib/.bbl.
# - Nếu bạn mới crawl xong 15.000 ID đầu và chưa crawl 10.000 ID cuối,
#   hãy thêm `--limit 15000` để tránh cào S2 trước cho 10k ID chưa có TeX:
#   !python run_crawl.py --ranges id_ranges.csv --limit 15000 --s2-only \
#       --enable-bib-fallback --s2-batch-size 150
!python run_crawl.py --ranges id_ranges.csv \
    --s2-only \
    --enable-bib-fallback \
    --s2-batch-size 150

In [ ]:
# ==============================================================================
# LOẠI 4: LIMIT 30 — CHẠY THỬ NGHIỆM 30 BÀI ĐẦU DẢI (PILOT / SMOKE TEST)
# ==============================================================================
# - Kiểm tra toàn diện kết nối, môi trường và chuỗi xử lý đầu-cuối với mẫu 30 bài.
!python run_crawl.py --ranges id_ranges.csv --limit 30 \
    --auto-resume-rate-limit \
    --enable-bib-fallback \
    --max-stream-mbps 6.0 \
    --http2 \
    --in-flight-extract

### Quy trình dừng / chạy tiếp (Pause & Resume)

**A. Dừng tạm thời nhưng runtime Colab vẫn còn sống** (khi muốn nghỉ hoặc hết giờ):
1. Bấm nút **Runtime → Interrupt execution** (dừng mềm). Crawler sẽ tự động hoàn tất bài đang tải dở và lưu bản backup cuối cùng vào `arxiv-data.tar.gz`.
2. Để chắc chắn dữ liệu được đồng bộ an toàn lên Google Drive trước khi đóng tab: chạy cell **Backup** bên dưới với `backup_now(unmount=True)`.
3. Nếu muốn chạy tiếp ngay trong cùng session: chỉ cần bấm chạy lại cell **Crawl** (không cần chạy lại Restore).

**B. Bị ngắt kết nối / Mất runtime (Disconnect)**:
1. Mở lại notebook, chạy lần lượt từ trên xuống:
   - Cell 1: Clone / Pull code.
   - Cell 2: Setup & Mount Drive.
   - Cell 5: Restore (`restore_now()`) — toàn bộ dữ liệu cào trước đó sẽ được giải nén trở lại `/content/data` trong vài giây.
   - Chạy cell Crawl tương ứng (ví dụ Loại 2 để tiếp tục cào).

**C. Ý nghĩa các mã thoát (Exit codes)**:
- `Exit code 0`: Đã hoàn thành 100% toàn bộ dải bài được giao.
- `Exit code 3`: arXiv đang tạm thời bận hoặc siết rate limit sau nhiều lần thử lại. Hệ thống đã tự động lưu dữ liệu an toàn. Hãy tạm dừng nghỉ 30–60 phút rồi chạy lại lệnh để tiếp tục.


In [ ]:
# Backup dữ liệu thủ công lên Drive (khuyên dùng unmount=True trước khi tắt máy)
backup_now()
# backup_now(unmount=True)

In [ ]:
# ==============================================================================
# KIỂM TRA & XUẤT BẢNG DỮ LIỆU (Chỉ chạy khi Giai đoạn 1 và 2 đã hoàn tất 100%)
# ==============================================================================

# 1. Kiểm tra tính toàn vẹn kho dữ liệu theo rubric đồ án (không thiếu file, không corrupt):
# !python crawler/validate_corpus.py --data-dir $CRAWLER_DATA_DIR

# 2. Kiểm toán đối soát kho tệp trên đĩa so với bảng manifest.sqlite:
# !python crawler/audit_corpus.py --data-dir $CRAWLER_DATA_DIR --output audit_report.json

# 3. Làm giàu trích dẫn offline từ các tệp .bib/.bbl có sẵn (không tốn mạng):
# !python crawler/enrich_references.py --data-dir $CRAWLER_DATA_DIR

# 4. Xuất dữ liệu ra bảng Parquet (papers.parquet, references.parquet) và thống kê:
# !python crawler/export_data.py --data-dir $CRAWLER_DATA_DIR